In [2]:
import glob


In [3]:
invalid_tags = linguistic_tags = [
    "bad",    # покруч/помилкове написання
    "rare",   # рідковживана форма
    "coll",   # розмовне слово/розмовна форма
    "arch",   # застаріле/архаїчне/діалектне
    "slang",  # сленг та (проф)жаргонізми
    "vulg",   # вульгарне
    "obsc",    # обсценне
    "subst"
]

In [4]:
def load_vesum(path="dic_data/dict_corp_lt.txt"):
    vesum = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 3:
                continue
            form, lemma, tags = parts

            is_good = set(invalid_tags).isdisjoint(tags.split(":"))
            if form.lower() not in vesum:
                vesum[form.lower()] = [{lemma}, set(tags.split(":")).intersection(set(invalid_tags)), is_good]
            else:
                vesum[form.lower()][0].add(lemma)
                if not is_good:
                    vesum[form.lower()][1].add(tags.split(":")[-1])
                else:
                    vesum[form.lower()][2] = True

    # convert all sets to lists after building
    for key in vesum:
        vesum[key][0] = list(vesum[key][0])
        vesum[key][1] = list(vesum[key][1])

    return vesum
def load_replacements(dict_dir="dic_data/replacements"):
    replacements = {}
    for path in glob.glob(f"{dict_dir}/*.lst"):
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue

                if "#>" not in line:
                    continue

                word_part, replacement_part = line.split("#>", 1)

                word = word_part.strip().split()[0]  # just the word, no flags

                # "поглинальний|вбиральний|абсорбівний|..."
                # split by | to get individual suggestions as a list
                suggestions = [s.strip() for s in replacement_part.strip().split("|")]

                if word:
                    replacements[word] = replacements.get(word, []) + suggestions

    return replacements

In [5]:
vesum = load_vesum()
replacements = load_replacements()



In [6]:
def query(word):
    entry = vesum.get(word)

    if not entry:
        return {"known": False}

    lemma, tags, is_possibly_good = entry

    result = {
        "known": True,
        "lemmas": lemma,
        "is_bad": not set(linguistic_tags).isdisjoint(tags),
        "is_possibly_good": is_possibly_good,
        "tags": tags,
    }

    return result

In [11]:
print(query("авралу"))

{'known': True, 'lemmas': ['аврал'], 'is_bad': False, 'is_possibly_good': True, 'tags': []}


In [ ]:
import os
import spacy
from pydantic import BaseModel, Field
from typing import Literal
from openai import AsyncOpenAI

nlp = spacy.load("uk_core_news_sm")
_key = os.environ.get("OPENAI_API_KEY", "")
if not _key:
    raise RuntimeError("Set OPENAI_API_KEY or run gradio_app.py and paste your key there.")
client = AsyncOpenAI(api_key=_key)

class DictEntry(BaseModel):
    lemma: str
    classification: Literal["slang", "borrowed", "archaic", "neologism", "bad"]
    replacements: list[str]
    word_forms: list[str]  # all possible forms LLM can generate
    origin: str
    meaning: str
    examples: list[str]
    added_at: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

class ParagraphResult(BaseModel):
    fixed_paragraph: str

In [8]:
import meilisearch

ms_client = meilisearch.Client("http://localhost:7700", api_key="masterKey123")
index = ms_client.index("custom_dict")

# configure once at startup
index.update_settings({
    "searchableAttributes": ["lemma", "word_forms"],
    "filterableAttributes": ["classification"],
    "typoTolerance": {
        "enabled": True,
        "minWordSizeForTypos": {
            "oneTypo": 4,
            "twoTypos": 8
        }
    },
    "prefixSearch": "disabled"
})


import uuid

def make_id() -> str:
    return str(uuid.uuid4())

async def add_to_meilisearch(entry: DictEntry):
    doc = entry.model_dump()
    doc["id"] = make_id()

    def _add():
        task = index.add_documents([doc])
        result = ms_client.wait_for_task(task.task_uid)

        return result

    return await asyncio.to_thread(_add)

import asyncio

async def lookup_custom_dict(word: str, limit: int = 3) -> list[dict]:
    return await asyncio.to_thread(_lookup_sync, word, limit)


def _lookup_sync(word: str, limit: int) -> list[dict]:
    results = index.search(word, {
        "limit": limit,
        "attributesToSearchOn": ["lemma", "word_forms"]
    })
    return results.get("hits", [])

In [10]:
index.delete_documents(["fd633170-4469-46e2-a1eb-0fdd20ecec4c"])

TaskInfo(task_uid=711, index_uid='custom_dict', status='enqueued', type='documentDeletion', enqueued_at=datetime.datetime(2026, 4, 2, 16, 45, 23, 486648))

In [12]:
from datetime import  timedelta

rejected_index = ms_client.index("rejected_words") # temporary, auto-cleaned
# configure rejected index
rejected_index.update_settings({
    "searchableAttributes": ["word", "reason"],
    "filterableAttributes": ["expires_at", "word"],
    "sortableAttributes": ["expires_at"]
})

REJECTION_TTL_DAYS = 30

async def add_to_rejected(word: str, skip_reason: str):
    expires_at = (datetime.now() + timedelta(days=REJECTION_TTL_DAYS)).isoformat()

    await asyncio.to_thread(
        rejected_index.add_documents, [{
            "id": make_id(),
            "word": word,
            "skip_reason": skip_reason,
            "rejected_at": datetime.now().isoformat(),
            "expires_at": expires_at,
        }]
    )

async def is_rejected(word: str) -> bool:
    try:
        results = await asyncio.to_thread(
            rejected_index.search, word, {
                "filter": f"word = '{word}' AND expires_at > '{datetime.now().isoformat()}'",
                "attributesToSearchOn": ["word"],
                "limit": 1
            }
        )
        return len(results["hits"]) > 0
    except:
        return False

In [13]:
import re

URL_REGEX = re.compile(r'https?://\S+|www\.\S+')

def is_url(word: str) -> bool:
    return bool(URL_REGEX.match(word))

TECH_REGEX = re.compile(r"""
    ^[a-zA-Z0-9\.-]+\.(com|net|org|io|js|py|dev)$ |  # domains
    ^[a-zA-Z][a-zA-Z0-9_\.-]*$                      # latin tokens
""", re.VERBOSE)

def is_tech_token(word: str) -> bool:
    return bool(TECH_REGEX.match(word))

def is_mixed(word: str) -> bool:
    return any(c.isalpha() for c in word) and any(c.isdigit() for c in word)

def has_special_tech_chars(word: str) -> bool:
    return any(c in word for c in "+#.")

def is_named_entity_like(word: str) -> bool:
    return (
        is_url(word) or
        is_mixed(word) or
        is_tech_token(word) or
        has_special_tech_chars(word) or
        any('a' <= c.lower() <= 'z' for c in word)  # latin
    )

In [14]:
import emoji

def remove_emojis(text: str) -> str:
    return emoji.replace_emoji(text, replace="")

def normalize_apostrophes(text: str) -> str:
    return (
        text
        .replace("’", "'")
        .replace("`", "'")
        .replace("ʼ", "'")
        .replace("ʹ", "'")
    )

def normalize_text(text: str) -> str:
    text = remove_emojis(text)
    text = normalize_apostrophes(text)
    return text

In [20]:
async def extract_candidates(paragraph: str, account_for_untypical_usage: bool = True) -> dict:  # no more custom_dict param
    paragraph = normalize_text(paragraph)
    doc = nlp(paragraph)
    named_entities = {ent.text.lower() for ent in doc.ents}

    words = set()
    for token in doc:
        if token.is_punct or token.is_space or token.like_num:
            continue
        words.add(token.text.lower())

    result = {
        "entities": [],
        "standard": [],
        "known_bad": [],
        "known_bad_no_repl": [],
        "in_custom_dict": [],
        "unknown": [],
    }

    for word in words:
        clean = word.strip(".,!?;:")

        if clean in named_entities or is_named_entity_like(clean):
            result["entities"].append(clean)
            continue

        vesum_result = query(clean)
        if vesum_result["known"] and not vesum_result["is_bad"] :
            result["standard"].append(clean)
            continue

        # run rejected + custom dict check concurrently
        rejected, custom_matches = await asyncio.gather(
            is_rejected(clean),
            lookup_custom_dict(clean)
        )

        if rejected:
            result["standard"].append(clean)
            continue

        if custom_matches:

            result["in_custom_dict"].append({
                "word": clean,
                "entries": [{"replacements": w["replacements"], "meaning": w["meaning"], "examples": w["examples"]} for w in custom_matches],
            })
            continue



        if not vesum_result["known"]:
            result["unknown"].append(clean)
            continue

        if vesum_result["is_bad"] and not (vesum_result["is_possibly_good"] and not account_for_untypical_usage):
            lemmas = vesum_result.get("lemmas", [])

            word_replacements = list({
                repl
                for lemma in lemmas
                for repl in replacements.get(lemma, [])
            })

            entry = {
                "word": clean,
                "vesum": vesum_result,
                "replacements": word_replacements
            }

            if word_replacements:
                result["known_bad"].append(entry)
            else:
                result["known_bad_no_repl"].append(entry)

            continue

        result["standard"].append(clean)

    return result

In [26]:
candidates = await extract_candidates("""цей гад аоврілоарівола згадав, шо мона гроші списувать
""")
print(f"fcandidates: {candidates["unknown"] + [w["word"] for w in candidates["known_bad"]] + [w["word"] for w in candidates["known_bad_no_repl"]] + [w["word"] for w in candidates["in_custom_dict"]]}")

fcandidates: ['аоврілоарівола', 'шо', 'мона']


In [33]:
LOG_FILE = "llm_search_logs.jsonl"

def log_event(event: dict):
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(event, ensure_ascii=False) + "\n")

In [70]:
import json
from pydantic import BaseModel

def json_dumps(obj) -> str:
    def default(o):
        if isinstance(o, set):
            return list(o)
        if isinstance(o, BaseModel):
            return o.model_dump()
        if hasattr(o, '__dict__'):
            return o.__dict__
        raise TypeError(f"Not serializable: {type(o)}")

    return json.dumps(obj, ensure_ascii=False, default=default)

In [72]:
import asyncio
import time
from datetime import datetime, timezone


# extended model to handle the "not worth adding" case
class SearchResult(BaseModel):
    worth_adding: bool          # model decides if this deserves a dict entry
    skip_reason: str | None     # if not worth adding — why
    leave_as_is: bool           # tell sentence fixer to leave word unchanged

    # only populated if worth_adding = True
    lemma: str | None = None
    classification: Literal["slang", "borrowed", "archaic", "neologism", "bad"] | None = None
    replacements: list[str] = []
    word_forms: list[str] = []
    origin: str | None = None
    meaning: str | None = None
    examples: list[str] = []
    added_at: str = Field(default_factory=lambda: datetime.now().isoformat())

def extract_sentence_with_neighbors(text: str, word: str) -> str:
    sentences = re.split(r'(?<=[.!?])\s+', text)

    for i, s in enumerate(sentences):
        if word.lower() in s.lower():
            context = [s]

            # add previous sentence
            if i > 0:
                context.insert(0, sentences[i - 1])

            # add next sentence
            if i < len(sentences) - 1:
                context.append(sentences[i + 1])

            return " ".join(context).strip()

    return text[:200]

async def search_unknown_words(
    unknown_words: list,
    no_repl_words: list,
    paragraph: str,
    *,
    max_concurrent: int = 20,
) -> dict:

    words_to_search = []

    for word in unknown_words:
        words_to_search.append({
            "word": word,
            "type": "unknown",
            "possible_lemmas": [],
            "is_possibly_good": False,
            "vesum_tags": None
        })

    for w in no_repl_words:
        words_to_search.append({
            "word": w["word"],
            "type": "known_bad",
            "possible_lemmas": w["vesum"]["lemmas"],
            "is_possibly_good": w["vesum"]["is_possibly_good"],
            "vesum_tags": w["vesum"]["tags"]
        })

    if not words_to_search:
        return {}

    sem = asyncio.Semaphore(max_concurrent)

    async def search_one(item):
        async with sem:
            return await _search_one_word(item)

    async def _search_one_word(item):
        word = item["word"]      # fix — was word["type"] before
        context = extract_sentence_with_neighbors(paragraph, word)
        tags = item["vesum_tags"]
        is_possibly_good = item["is_possibly_good"]
        possible_lemmas = item["possible_lemmas"]
        word_type = item["type"] # fix — separate variable

        start_time = time.time()

        try:
            response = await client.responses.parse(
                model="gpt-5.4",
                tools=[{
                    "type": "web_search",
                }],
                temperature=0.1,
                tool_choice="required",
                text_format=SearchResult,
                input=f"""
You are analyzing a Ukrainian word for inclusion in a custom dictionary of slang, borrowed, or non-standard words.
When searching, prefer results from:
- reddit.com
- goroh.pp.ua
- myslovo.com
- dou.ua
- twitter.com
- wikipedia.org
Word: "{word}"

Metadata:
- Source: VESUM dictionary
- Tags: {tags}
- Possible lemmas: {possible_lemmas}
- Possibly standard/acceptable: {is_possibly_good}
- Word type: {word_type}

Context:
The word appears in the following paragraph:
"{context}"

Instructions:

1. Determine the meaning of the word strictly based on:
   - the provided context
   - real-world usage (via search)

2. Use the web search tool to:
   - find real usage examples of "{word}" (Ukrainian or Russian, Cyrillic)
   - extract 1–3 full example sentences (no links)
   - include the provided paragraph as one example

3. Infer the meaning from actual usage. Do NOT guess if evidence is weak.

4. Decide whether the word is worth adding to the dictionary:

   Mark as NOT worth adding if:
   - it is a standard Ukrainian word
   - it is a named entity (person, place, brand, etc.)
   - it has no clear or consistent meaning
   - it is noise or malformed text

   Mark as worth adding if:
   - it is slang, borrowed, or non-standard
   - it has clear meaning supported by multiple usages

5. Special rule:
   - If `is_possibly_good = True` AND the word is a known standard Ukrainian form → mark as NOT worth adding

6. If the word IS worth adding:
   - provide a clear meaning
   - suggest the best standard Ukrainian replacement (if applicable)
   - Return all inflected forms of the word (same lemma only). Do not include phrases, prepositions, or derived words.

7. If the word is NOT worth adding:
   - explain why briefly
   - set `leave_as_is = true`

Rules:
- Base conclusions ONLY on evidence (context + search)
- Do NOT hallucinate meanings or examples
- Prefer rejecting over guessing
- Be concise and factual

"""
            )

            duration = time.time() - start_time

            # parse structured output from response text
            try:
                raw = json.loads(response.output_text)
                search_result = SearchResult(**raw)
            except Exception:
                # if parsing fails treat as uncertain — flag for review
                search_result = SearchResult(
                    worth_adding=False,
                    skip_reason="failed to parse model response",
                    leave_as_is=True
                )

            # save to meilisearch if model thinks it's worth it
            if search_result.worth_adding:
                entry = DictEntry(
                    lemma=search_result.lemma,
                    classification=search_result.classification,
                    replacements=search_result.replacements,
                    word_forms=search_result.word_forms,
                    origin=search_result.origin,
                    meaning=search_result.meaning,
                    examples=search_result.examples,
                )
                await add_to_meilisearch(entry)
                log_event({
                    "timestamp": datetime.now(timezone.utc).isoformat(),
                    "word": word,
                    "status": "success",
                    "duration_sec": round(duration, 3),
                    "worth_adding": search_result.worth_adding,
                    "leave_as_is": search_result.leave_as_is,
                    "output_text": response.output_text,
                })
            else:
                # not worth adding — log why
                log_event({
                    "timestamp": datetime.now(timezone.utc).isoformat(),
                    "word": word,
                    "status": "rejected",
                    "leave_as_is": search_result.leave_as_is,
                    "reason": search_result.skip_reason,
                })

                await add_to_rejected(word, search_result.skip_reason)



            # return result so analyze_paragraph knows how to handle this word
            return word, {
                "search_result": search_result,
                "output_text": response.output_text,
            }

        except Exception as e:
            duration = time.time() - start_time

            log_event({
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "word": word,
                "status": "error",
                "duration_sec": round(duration, 3),
                "error": str(e),
            })

            return word, None

    tasks = [search_one(item) for item in words_to_search]
    results_list = await asyncio.gather(*tasks)

    return {word: result for word, result in results_list}

In [87]:
class WordNeedsSearch(BaseModel):
    word: str
    reason: str  # why existing info is insufficient

class AnalyzeOnlyAnalysis(BaseModel):
    words_needing_clarification: list[WordNeedsSearch]  # no fixed_paragraph

class ParagraphAnalysis(BaseModel):
    fixed_paragraph: str
    words_needing_clarification: list[WordNeedsSearch]

async def analyze_paragraph(
    paragraph: str,
    account_for_untypical_usage: bool = False,
    fix_paragraph: bool = True,

) -> ParagraphResult:

    candidates = await extract_candidates(paragraph, account_for_untypical_usage)

    search_results = await search_unknown_words(
        candidates["unknown"],
        candidates["known_bad_no_repl"],
        paragraph
    )

    needs_llm = (
        candidates["unknown"] or
        candidates["known_bad"] or
        candidates["known_bad_no_repl"] or
        candidates["in_custom_dict"]
    )

    if not needs_llm:
        return ParagraphResult(
            fixed_paragraph=paragraph,
        )

    # system prompt stays the same — behavior changes via schema
    system_content = """You are a Ukrainian language expert.

For each non-standard word:
1. If word comes from search results — it already fits context, replacement is appropriate
2. If word is marked leave_as_is=True — leave it unchanged
3. If the word has replacements from the custom dictionary:
   - Evaluate each replacement using the provided meaning and examples of the original word.
   - Select the replacement that best matches the meaning in this specific context.
   - If no replacement clearly fits the context, add the word to words_needing_clarification with a short reason.

4. If the word has replacements from VESUM:
   - Evaluate each replacement in the given context.
   - Select the most contextually appropriate replacement.
   - If no replacement clearly fits the context, add the word to words_needing_clarification with a short reason.


When multiple replacements exist pick the most contextually appropriate one.
Use correct grammatical forms for all replacements.
Use Ukrainian language only.
Do NOT invent replacements — only use provided ones.
"""

    # add fix instruction only when needed — saves output tokens
    if fix_paragraph:
        system_content += "\nReturn the fully fixed paragraph in fixed_paragraph field. Leave the words_needing_clarification as is."
    else:
        system_content += "\nDo NOT output a fixed paragraph — only flag words that need clarification or have contextual issues. This saves tokens."

    messages = [
        {"role": "system", "content": system_content},
        {
            "role": "user",
            "content": f"""
Paragraph: {paragraph}

Words in our custom dictionary with known meanings and replacements:
{json_dumps(candidates["in_custom_dict"])}

Non-standard words with VESUM suggested replacements (list — pick best for context):
{json_dumps(candidates["known_bad"])}

Non-standard words with no replacements + unknown words:
{json_dumps(candidates["known_bad_no_repl"] + candidates["unknown"])}

Search results from web (already context-appropriate):
{json_dumps(search_results)}
"""
        }
    ]

    iteration = 0
    analysis = None

    # pick schema based on mode
    output_format = ParagraphAnalysis if fix_paragraph else AnalyzeOnlyAnalysis

    result = await client.responses.parse(
        model="gpt-5.4",
        temperature=0.3,
        input=messages,
        text_format=output_format  # ← different schema per mode
    )

    analysis = result.output_parsed

    if not analysis.words_needing_clarification:
        print(f"Resolved in {iteration} iteration(s)")
        if analysis is None:
            return ParagraphResult(
                fixed_paragraph=paragraph
            )

        return ParagraphResult(
            # use fixed paragraph if available, otherwise return original unchanged
            fixed_paragraph=analysis.fixed_paragraph if fix_paragraph else paragraph,
        )

    flagged = [w.word.lower() for w in analysis.words_needing_clarification]
    print(f"Iteration {iteration}: needs clarification for {flagged}")


    new_results = await search_unknown_words(
        unknown_words=flagged,
        no_repl_words=[],
        paragraph=paragraph
    )

    if fix_paragraph:


        messages[-1] = {
                "role": "user",
                "content": f"""
                Partially fixed paragraph: {analysis.fixed_paragraph}
                Search results from web for marked words (already context-appropriate):
                {json_dumps(new_results)}

                Fix the partially fixed paragraph using new search results and return the completely fixed paragraph.
                """
            }
        result = await client.responses.parse(
            model="gpt-5.4",
            input=messages,
            text_format=ParagraphResult  # ← different schema per mode
        )

        analysis = result.output_parsed

    if analysis is None:
        return ParagraphResult(
            fixed_paragraph=paragraph
        )

    return ParagraphResult(
        # use fixed paragraph if available, otherwise return original unchanged
        fixed_paragraph=analysis.fixed_paragraph if fix_paragraph else paragraph,
    )

In [75]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="https://huggingface.co/datasets/lang-uk/Reddit-MultiGEC/resolve/main/reddit_uk_annotations.csv",
    split="train"
)

In [76]:
def batch_iterator(dataset, batch_size):
    batch = []
    for row in dataset:
        batch.append(row)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

def get_paragraph(row):
    return row["text"]

In [90]:
from tqdm import tqdm

async def process_dataset(dataset):
    total = len(dataset) // 20 + 1  # approximate

    pbar = tqdm(total=total, desc="Processing batches")

    for batch in batch_iterator(dataset, 20):
        paragraphs = "\n".join([get_paragraph(row) for row in batch])
        await analyze_paragraph(paragraphs, fix_paragraph=False, account_for_untypical_usage=False)
        pbar.update(1)

    pbar.close()

In [89]:
print(await process_dataset(dataset))


Processing batches:   0%|          | 0/76 [00:22<?, ?it/s]

Resolved in 0 iteration(s)
fixed_paragraph='Всім привіт, у мене є проблема з ноутом. Виблимував він, очевидно, проблему батареї. Потім ще й текстом підтвердив. Варіантів три: або в батареї поїхав дах і здурів термодатчик, або в корпусі занадто спекотно, або в батареї чомусь грілася одна з банок. Яка зношеність батареї? І чи рідна вона? Відкрити, глянути. Делл ду-уже чутливі до оригінальності і якості батарей.\nЧи поганка я через те, що не хочу ділитися? Це, звісно, так, але я ділю кімнату із сестрою. Якби була своя кімната, звісно, їла б так, щоб ніхто не бачив.\nЯ жінка і ненавиджу своє тіло через менструацію. Чи нормально це? Підкажіть, хто може мені допомогти. Я не хочу війни з власним тілом. Як варіант, можна розглянути оральні контрацептиви, існують варіанти пролонгованого прийому до трьох місяців, та й навіть зі щомісячними паузами значно знижується інтенсивність кровотечі та больових відчуттів. Для мене стало набагато зручніше жити, і я не випадаю з життя на 5 днів підряд.\nПауз

In [122]:
paragraph = """
Що ви знаєте про .NET? Можливості цієї платформи - неймовірно великі, тож не варто намагатися охопити їх усі й одразу. Краще зосередитися на специфічних аспектах, залежно від обраного напряму розробки.

Ми зібрали для вас наші статті та пости, які допоможуть розібратися та почати вивчати .NET
"""

result = await analyze_paragraph(paragraph)

print(result.fixed_paragraph)
# "Він працює бекенд-розробником у стартапі і постійно
#  знущається з тімліда через те що розгортання зазнало невдачі вчора."


# custom_dict now has new entries saved automatically

Resolved in 1 iteration(s)
Що ви знаєте про .NET? Можливості цієї платформи — неймовірно великі, тож не варто намагатися охопити їх усі й одразу. Краще зосередитися на специфічних аспектах, залежно від обраного напряму розробки.

Ми зібрали для вас наші статті та дописи, які допоможуть розібратися та почати вивчати .NET


In [191]:
index = ms_client.index("custom_dict")
index.delete_all_documents()



TaskInfo(task_uid=201, index_uid='custom_dict', status='enqueued', type='documentDeletion', enqueued_at=datetime.datetime(2026, 3, 29, 3, 29, 18, 586383))

In [54]:
print(index.get_stats())

number_of_documents=7 is_indexing=False field_distribution=<meilisearch.models.index.FieldDistribution object at 0x000001E04EF35CD0>


In [42]:
print(type(client.responses))

<class 'openai.resources.responses.responses.AsyncResponses'>


In [43]:
!pip uninstall openai -y
!pip install --upgrade openai

Found existing installation: openai 2.30.0
Uninstalling openai-2.30.0:
  Successfully uninstalled openai-2.30.0
  Using cached openai-2.30.0-py3-none-any.whl.metadata (29 kB)
Using cached openai-2.30.0-py3-none-any.whl (1.1 MB)



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
